### Step 1 - Read csv file

In [0]:
%run ../00-common/01.environment_config

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/02.bronze_helpers

In [0]:
sources_path = f"{landing_folder_path}/{v_batch_id}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType,DateType

In [0]:
schema=StructType([
    StructField('season',IntegerType()),
    StructField('round',IntegerType()),
    StructField('url',StringType()),
    StructField('raceName',StringType()),
    StructField('date',DateType()),
    StructField('circuitId',StringType())

    ]
)

In [0]:
races_df=(
    spark.read.format('csv')
    .option('header',True)
    #.option('inferSchema',True)
    .option('mode','FAILFAST')
    .schema(schema)
    .load('/Volumes/formula1/landing/files/races.csv'))

### Step 2 - Ingest time stamp and file path

In [0]:
from pyspark.sql.functions import *

In [0]:
races_final_df = add_ingestion_metadata(races_df)

### Step3- Write data to Bronze delta table

In [0]:
write_data_to_bronze(races_final_df,table_name, v_batch_id)

In [0]:
%sql
--drop table formula1_incr.bronze.races;

In [0]:
%sql
select * from formula1_incr.bronze.races;